# Alphamind新手入门之七：单因子分析

In [ ]:
import asyncio
import datetime as dt


## 1. 准备因子历史数据
-----------------------

In [ ]:
import numpy as np
import pyarrow as pa
import pyarrow.compute as pc


In [ ]:
# 相关数据的获取
# 包括因子值获取、风险模型数据、股票收益率数据
rng = np.random.default_rng(42)
n_dates = 6


## 2. 计算历史IC序列
----------------------

In [ ]:
# 因子中性化
# 对neutralized_styles的因子进行中性化
n_assets = 12
dates = [dt.datetime(2026, 1, 5) + dt.timedelta(days=index) for index in range(n_dates)]


In [ ]:
codes = np.array([f"{index:06d}" for index in range(1, n_assets + 1)])
n_rows = n_dates * n_assets
panel = pa.table({
    "trade_date": [value for value in dates for _ in range(n_assets)],
    "code": np.tile(codes, n_dates),
    "factor": rng.normal(size=n_rows),
    "value": rng.normal(size=n_rows),
    "dx": rng.normal(0.0, 0.01, size=n_rows),
    "industry": np.tile(np.array(["Bank", "Auto", "Health"]), n_dates * n_assets // 3),
    "benchmark_weight": np.full(n_rows, 1.0 / n_assets),
    "BETA": np.tile(np.linspace(0.8, 1.2, n_assets) ** 2, n_dates),
    "SIZE": np.tile(np.linspace(-1.0, 1.0, n_assets), n_dates),
})


## 3. 行业内IR计算
---------------------

In [ ]:
# 获取行业数据
import calc_flow as cf
from alphamind.data.standardize import standardize


In [ ]:
# 行业IC的计算
from alphamind.data.winsorize import winsorize_normal

signal = panel["factor"].to_numpy(zero_copy_only=False).reshape(-1, 1)


In [ ]:
# 行业IR的计算ic.mean/ic.std
processed = standardize(winsorize_normal(signal)).flatten()
normalized = panel.set_column(
    panel.schema.get_field_index("factor"), "factor", pa.array(processed)
)
ic = await asyncio.to_thread(
    cf.compute,
    normalized,
    lambda input_table: input_table.sql("""
        SELECT trade_date, CORR(factor, dx) AS ic
        FROM input GROUP BY trade_date ORDER BY trade_date
    """),
)


In [ ]:
assert ic.num_rows == n_dates
ic
